In [2]:
import requests as req
from bs4 import BeautifulSoup as bs
from urllib.parse import urljoin
import time
import csv

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}
session = req.Session()
session.headers.update(headers)
processed_urls = set()  # Track processed URLs to avoid duplicates

def print_header(title):
    print(f"\n{'=' * 50}")
    print(f"{title.upper():^50}")
    print(f"{'=' * 50}")

def print_status(message, status="INFO"):
    print(f"[{status}] {message}")

In [4]:
def analyze_structure(url):
    try:
        response = session.get(url)
        response.raise_for_status()
        soup = bs(response.text, 'lxml')

        print_header("HTML Structure Analysis")
        print(f"Page Title: {soup.title.text if soup.title else 'Not available'}")

        # Focus on main article containers
        main_containers = soup.select('article.list-content__item')
        print(f"\nMain Article Containers Found: {len(main_containers)}")

        if main_containers:
            sample = main_containers[0]
            print("\nSample Container Analysis:")
            print(f"- CSS Classes: {', '.join(sample.get('class', ['None']))}")

            # Find the main link in each article
            main_link = sample.select_one('h2.media__title a') or sample.select_one('a.media__link')
            if main_link:
                print(f"- Article URL: {main_link.get('href', 'Not available')}")
                print(f"- Title Preview: {main_link.get_text(strip=True)[:50]}...")

        return soup

    except Exception as e:
        print_status(f"Structure analysis failed: {e}", "ERROR")
        return None

In [5]:
def find_unique_articles(soup):
    articles = []

    article_containers = soup.select('article.list-content__item')
    print_status(f"Processing {len(article_containers)} article containers")

    for container in article_containers:
        try:
            # Find the main article link and title
            title_link = (
                container.select_one('h2.media__title a') or
                container.select_one('h3 a') or
                container.select_one('a[href*="/news/"]') or
                container.select_one('a[href*="/sport/"]') or
                container.select_one('a[href*="/finance/"]') or
                container.select_one('a[href*="/hot/"]') or
                container.select_one('a[href*="/edu/"]') or
                container.select_one('a[href*="/health/"]') or
                container.select_one('a[href*="/jatim/"]')
            )

            if not title_link or not title_link.get('href'):
                continue

            # Get URL and normalize it
            url = title_link.get('href')
            if url.startswith('/'):
                url = urljoin('https://www.detik.com', url)
            elif not url.startswith('http'):
                continue

            # Skip if already processed (avoid duplicates)
            if url in processed_urls:
                continue

            # Get title
            title = title_link.get_text(strip=True)
            if not title or len(title) < 10:  # Skip very short titles
                continue

            # Get date from container
            date_elem = container.select_one('span.media__date') or container.select_one('[class*="date"]')
            date = date_elem.get_text(strip=True) if date_elem else 'Not available'

            # Clean date
            date = date.replace('WIB', '').replace('detikNews', '').strip()
            if ',' in date:
                date = date.split(',')[-1].strip()

            # Add to processed URLs
            processed_urls.add(url)

            articles.append({
                'title': title,
                'url': url,
                'date': date,
                'container': container
            })

        except Exception as e:
            print_status(f"Error processing container: {e}", "WARNING")
            continue

    return articles

In [6]:
def extract_article_content(url):
    try:
        time.sleep(1)  # Rate limiting
        response = session.get(url)
        response.raise_for_status()

        soup = bs(response.text, 'lxml')

        # Cari konten dengan berbagai selector (in priority order)
        content_selectors = [
            'div.detail__body-text.itp_bodycontent',  # Most specific first
            'div[class*="detail__body"]',
            'div[class*="itp_bodycontent"]',
            'div[class*="content-body"]',
            'div[class*="article-body"]',
            '.detail-text',
            '.content-text'
        ]

        content = ""
        content_found = False

        for selector in content_selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements:
                    paragraphs = elem.find_all('p')
                    if paragraphs:  # Only if paragraphs found
                        for p in paragraphs:
                            text = p.get_text(strip=True)
                            if text and len(text) > 20:  # Skip very short paragraphs
                                content += text + " "
                        content_found = True
                        break
                if content_found:
                    break

        # Fallback: ambil paragraf dari body utama
        if not content:
            main_content = soup.select_one('.detail-in') or soup.select_one('main') or soup
            if main_content:
                paragraphs = main_content.find_all('p')
                for p in paragraphs[:8]:  # Limit to first 8 paragraphs
                    text = p.get_text(strip=True)
                    if text and len(text) > 20:
                        content += text + " "

        # Clean content
        content = content.replace('\n', ' ').replace('\t', ' ')
        content = ' '.join(content.split())  # Remove extra spaces

        # Remove common unwanted text
        unwanted_phrases = [
            'ADVERTISEMENT', 'SCROLL TO RESUME CONTENT',
            'Baca juga:', 'Simak Video', 'BACA JUGA:',
            'Halaman selanjutnya', 'Lanjutkan membaca'
        ]

        for phrase in unwanted_phrases:
            content = content.replace(phrase, '')

        content = content.strip()

        return content[:7000]  # Limit content length

    except Exception as e:
        print_status(f"Content extraction failed for {url}: {e}", "ERROR")
        return ""

In [7]:
def scrape_search_results(query='pemilu 2024', pages=3):
    global processed_urls
    all_articles = []
    processed_urls.clear()  # Reset processed URLs

    print_header(f"Scraping Detik.com for: '{query}'")
    print(f"Pages to scrape: {pages}")

    for page in range(1, pages + 1):
        print_status(f"Processing page {page}/{pages}")
        url = f'https://www.detik.com/search/searchnews?query={query}&sortby=time&page={page}'

        try:
            response = session.get(url)
            response.raise_for_status()

            soup = bs(response.text, 'lxml')

            # Analyze structure only on first page
            if page == 1:
                analyze_structure(url)

            # Find unique articles
            articles = find_unique_articles(soup)
            print_status(f"Found {len(articles)} valid articles")

            for i, article in enumerate(articles):
                try:
                    print_status(f"Processing article {i+1}/{len(articles)}: {article['title'][:60]}...")

                    # Extract full content
                    content = extract_article_content(article['url'])

                    if content and len(content) > 50:  # Only save articles with substantial content
                        full_article = {
                            'title': article['title'],
                            'date': article['date'],
                            'url': article['url'],
                            'content': content
                        }

                        all_articles.append(full_article)
                        print_status(f"Article saved ({len(content)} characters)", "SUCCESS")
                    else:
                        print_status("Skipped - insufficient content", "WARNING")

                except Exception as e:
                    print_status(f"Processing failed: {e}", "ERROR")
                    continue

            print_status(f"Page {page} completed. Total articles: {len(all_articles)}")
            time.sleep(2)  # Delay between pages

        except Exception as e:
            print_status(f"Page {page} failed: {e}", "ERROR")
            continue

    return all_articles

In [ ]:
def save_to_csv(articles, filename='detik_articles.csv'):
    try:
        # Remove any remaining duplicates based on URL
        unique_articles = []
        seen_urls = set()

        for article in articles:
            if article['url'] not in seen_urls:
                unique_articles.append(article)
                seen_urls.add(article['url'])

        with open(filename, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Title', 'Date', 'URL', 'Content'])

            for article in unique_articles:
                writer.writerow([
                    article['title'],
                    article['date'],
                    article['url'],
                    article['content']
                ])

        print_header("Scraping Results Summary")
        print(f"Unique articles saved: {len(unique_articles)}")
        print(f"Duplicate articles removed: {len(articles) - len(unique_articles)}")
        print(f"Output file: {filename}")

    except Exception as e:
        print_status(f"Failed to save CSV: {e}", "ERROR")

In [9]:
def print_summary(articles):
    if not articles:
        print_status("No articles found!", "WARNING")
        return

    print_header("Scraping Summary")
    print(f"Total articles collected: {len(articles)}")
    print(f"Unique URLs: {len(set(a['url'] for a in articles))}")

    print("\nSample Articles:")
    for i, article in enumerate(articles[:3]):
        print(f"\nArticle {i+1}:")
        print(f"Title: {article['title'][:80]}...")
        print(f"Date: {article['date']}")
        print(f"URL: {article['url']}")
        print(f"Content Preview: {article['content'][:100]}...")
        print(f"Content Length: {len(article['content'])} characters")

In [11]:
if __name__ == "__main__":
    print("Starting Detik.com scraper (Fixed - No Duplicates)...")
    articles = scrape_search_results(query='uu+tni', pages=36)

    if articles:
        save_to_csv(articles)
        print_summary(articles)
    else:
        print("No articles found. Check the structure analysis output for debugging.")

Starting Detik.com scraper (Fixed - No Duplicates)...

         SCRAPING DETIK.COM FOR: 'UU+TNI'         
Pages to scrape: 36
[INFO] Processing page 1/36

             HTML STRUCTURE ANALYSIS              
Page Title: detiksearch

Main Article Containers Found: 10

Sample Container Analysis:
- CSS Classes: list-content__item
- Article URL: https://news.detik.com/berita/d-7948448/ibas-tekankan-pentingnya-kolaborasi-pemerintah-tni-kawal-program-mbg
- Title Preview: ...
[INFO] Processing 10 article containers
[INFO] Found 10 valid articles
[INFO] Processing article 1/10: Ibas Tekankan Pentingnya Kolaborasi Pemerintah & TNI Kawal P...
[SUCCESS] Article saved (5619 characters)
[INFO] Processing article 2/10: Oknum TNI AL Pembunuh Juwita Jurnalis di Kalsel Dituntut Pen...
[SUCCESS] Article saved (1861 characters)
[INFO] Processing article 3/10: Prajurit TNI AL Pembunuh Jurnalis Juwita Dituntut Bui Seumur...
[SUCCESS] Article saved (2006 characters)
[INFO] Processing article 4/10: Oknum TNI A

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv('/content/detik_articles.csv')

def parse_date(date_str):
    try:
        return pd.to_datetime(date_str, format='%d %b %Y %H:%M')
    except:
        return pd.Timestamp.now()

df['Date'] = df['Date'].apply(parse_date)
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M')
df.sample(100, random_state=42)


,Title,Date,URL,Content
194,Massa Tolak Revisi UU TNI Demo di Gerbang Depa...,2025-03-20 11:58,https://news.detik.com/berita/d-7832514/massa-...,Massa yang menolak revisi Undang-UndangTNImela...
157,Demo Tolak UU TNI Memanas hingga Fasilitas di ...,2025-03-25 03:30,https://www.detik.com/jatim/berita/d-7839809/d...,Aksi demo menolak UU TNI depan Gedung Grahadi ...
225,"Demo Tolak UU TNI di Kota Kediri, Petugas Dile...",2025-03-27 22:30,https://www.detik.com/jatim/berita/d-7845252/d...,Demo menolak UU TNI juga terjadi di Kota Kedir...
208,Puan ke Massa Demo Tolak Revisi UU TNI: Kami S...,2025-03-20 12:59,https://news.detik.com/berita/d-7832652/puan-k...,Ketua DPR RIPuan Maharanimenanggapi demonstras...
319,Video Said PDIP soal Usulan Pemakzulan Gibran:...,2025-06-04 11:05,https://20.detik.com/detikupdate/20250604-2506...,Ketua DPP PDIP Said Abdullah merespons surat p...
...,...,...,...,...
179,Satpol PP Buka Suara Usai Bongkar Tenda Massa ...,2025-04-10 10:01,https://news.detik.com/berita/d-7862046/satpol...,Satuan Polisi Pamong Praja (Satpol PP) Jakarta...
143,"Buka-bukaan Prabowo: UU TNI, Komunikasi, hingg...",2025-04-08 07:17,https://news.detik.com/berita/d-7858500/buka-b...,PresidenPrabowoSubianto buka-bukaan soal sejum...
19,"Puluhan Mahasiswa Demo di DPRD Sumut, Tuntut P...",2025-03-26 18:27,https://www.detik.com/sumut/berita/d-7843171/p...,Puluhan mahasiswa di Medan melakukan aksi demo...
256,"Ricuh Mereda, Massa Demo Tolak UU TNI di DPRD ...",2025-03-21 01:52,https://www.detik.com/jogja/berita/d-7833959/r...,Massa Aliansi Jogja Memanggil akhirnya membuba...


In [7]:
import pandas as pd
from datetime import datetime

df = pd.read_csv('../../RAG/web/detik_articles.csv')
df

,Title,Date,URL,Content
0,Cerita Mahasiswa UII Didatangi OTK Usai Gugat ...,24 Mei 2025 19:09,https://www.detik.com/jogja/berita/d-7930735/c...,Tiga mahasiswa dari Fakultas Hukum Universitas...
1,Pernyataan Sikap Sivitas FH UII Usai 3 Mahasis...,26 Mei 2025 11:57,https://www.detik.com/jogja/berita/d-7932885/p...,Tiga mahasiswa dari Fakultas Hukum Universitas...
2,Oknum TNI AL Jumran Pembunuh Jurnalis Juwita D...,6 jam yang lalu,https://news.detik.com/berita/d-7947816/oknum-...,"Oknum anggota TNI AL,Jumran, dituntut penjara ..."
3,Mensesneg Ungkap UU TNI Sudah Diteken Prabowo,17 Apr 2025 13:06,https://news.detik.com/berita/d-7873184/menses...,Menteri Sekretaris Negara (Mensesneg) Prasetyo...
4,Jimly Kritik Masalah Komunikasi Pemerintah Ter...,31 Mar 2025 10:35,https://news.detik.com/berita/d-7849675/jimly-...,Mantan Ketua Mahkamah Konstitusi (MK)Jimly Ass...
...,...,...,...,...
338,"Mahasiswa Gugat UU TNI ke MK, Minta Presiden-D...",09 Mei 2025 13:46,https://news.detik.com/berita/d-7907213/mahasi...,"Mahasiswa Universitas Putera Batam, Hidayattud..."
339,Legislator PKS Sebut RUU TNI Atur Usia Pensiun...,28 Mei 2024 16:10,https://news.detik.com/berita/d-7361840/legisl...,"Anggota Baleg DPR RI FraksiPKS, Mardani Ali Se..."
340,Anjing Pelacak Dikerahkan Cari 4 Korban Tertim...,4 jam yang lalu,https://www.detik.com/jabar/cirebon-raya/d-794...,Proses pencarian korban longsor di area tamban...
341,"KSAL Masuki Usia Pensiun, TNI Masih Ikut Atura...",09 Apr 2025 12:31,https://news.detik.com/berita/d-7860691/ksal-m...,Kepala Staf Angkatan Laut (KSAL) Laksamana Muh...
